# Trayectoria IA — V2
## Predicción de riesgo de deserción + perfilamiento de estudiantes

Este notebook está pensado para dos usos:

1. **Entrenamiento:** entrenar y evaluar el modelo binario `Dropout` vs `No Dropout`, construir perfiles mediante clustering y guardar un único artefacto de producción.
2. **Integración con la página:** recibir un CSV previamente validado por la aplicación, convertir sus categorías de texto a los códigos del dataset UCI, calcular riesgo y perfil, y devolver un CSV enriquecido.

### Flujo de integración
`CSV validado por la página → procesar_csv() → CSV enriquecido`

La página es responsable de validar formato/tipos básicos. Este notebook vuelve a verificar columnas y categorías antes de inferir para evitar predicciones silenciosamente incorrectas.


In [1]:
# En Google Colab ejecutar una vez.
!pip install -q imbalanced-learn ucimlrepo joblib


In [2]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    f1_score, accuracy_score, precision_score, recall_score,
    silhouette_score
)
from sklearn.cluster import KMeans
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


# 1. Cargar el dataset original de Portugal

Se usa el dataset UCI **Predict Students' Dropout and Academic Success (ID 697)**.
El objetivo de Trayectoria es binario:

- `1`: estudiante que termina en `Dropout`.
- `0`: estudiante `Enrolled` o `Graduate`.

Los códigos numéricos originales se conservan para el entrenamiento.


In [3]:
dataset = fetch_ucirepo(id=697)

X = dataset.data.features.copy()
target_original = dataset.data.targets.squeeze().copy()

y = (target_original == "Dropout").astype(int)
FEATURES_MODELO = X.columns.tolist()

print("Dimensiones:", X.shape)
print("N.º variables:", len(FEATURES_MODELO))
print("\nDistribución objetivo:")
print(y.value_counts())
print(y.value_counts(normalize=True).round(4))


Dimensiones: (4424, 36)
N.º variables: 36

Distribución objetivo:
Target
0    3003
1    1421
Name: count, dtype: int64
Target
0    0.6788
1    0.3212
Name: proportion, dtype: float64


# 2. Separar train/test y balancear únicamente el entrenamiento

El conjunto de test permanece intacto. SMOTE se aplica solamente al conjunto de entrenamiento para evitar fuga de información.


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

smote = SMOTE(random_state=RANDOM_STATE)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

# La regresión logística sí necesita escalado.
scaler_log = StandardScaler()
X_train_bal_scaled = scaler_log.fit_transform(X_train_bal)
X_test_scaled = scaler_log.transform(X_test)

print("Train original:", y_train.value_counts().to_dict())
print("Train con SMOTE:", pd.Series(y_train_bal).value_counts().to_dict())
print("Test:", y_test.value_counts().to_dict())


Train original: {0: 2402, 1: 1137}
Train con SMOTE: {1: 2402, 0: 2402}
Test: {0: 601, 1: 284}


# 3. Entrenar modelos candidatos

Se comparan Regresión Logística y Random Forest. La V2 corrige la inconsistencia de la versión anterior: la Regresión Logística se entrena y evalúa siempre con datos escalados.


In [5]:
modelos = {
    "Regresión Logística": LogisticRegression(
        max_iter=3000, random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight=None
    )
}

modelos["Regresión Logística"].fit(X_train_bal_scaled, y_train_bal)
modelos["Random Forest"].fit(X_train_bal, y_train_bal)

resultados = {}

for nombre, modelo in modelos.items():
    X_eval = X_test_scaled if nombre == "Regresión Logística" else X_test
    pred = modelo.predict(X_eval)
    proba = modelo.predict_proba(X_eval)[:, 1]

    resultados[nombre] = {
        "F1_desercion": f1_score(y_test, pred),
        "ROC_AUC": roc_auc_score(y_test, proba),
        "Accuracy": accuracy_score(y_test, pred),
        "Precision_desercion": precision_score(y_test, pred),
        "Recall_desercion": recall_score(y_test, pred)
    }

    print(f"\n=== {nombre} ===")
    print(classification_report(y_test, pred, target_names=["No deserta", "Deserta"]))

comparacion = pd.DataFrame(resultados).T.sort_values("F1_desercion", ascending=False)
comparacion



=== Regresión Logística ===
              precision    recall  f1-score   support

  No deserta       0.89      0.91      0.90       601
     Deserta       0.79      0.77      0.78       284

    accuracy                           0.86       885
   macro avg       0.84      0.84      0.84       885
weighted avg       0.86      0.86      0.86       885


=== Random Forest ===
              precision    recall  f1-score   support

  No deserta       0.90      0.93      0.92       601
     Deserta       0.84      0.79      0.82       284

    accuracy                           0.88       885
   macro avg       0.87      0.86      0.87       885
weighted avg       0.88      0.88      0.88       885



,F1_desercion,ROC_AUC,Accuracy,Precision_desercion,Recall_desercion
Random Forest,0.815217,0.929797,0.884746,0.839552,0.792254
Regresión Logística,0.784314,0.920742,0.863277,0.794224,0.774648


# 4. Modelo de riesgo definitivo

Para mantener la integración estable y reproducible, la V2 utiliza **Random Forest** como modelo de producción. Esto coincide con el modelo seleccionado en el notebook original y evita que un cambio pequeño de métricas al reejecutar el notebook cambie inesperadamente el tipo de modelo que consume la página.

El valor importante para la integración es `probabilidad_desercion`, obtenido con `predict_proba`.


In [6]:
modelo_riesgo = modelos["Random Forest"]

proba_full = modelo_riesgo.predict_proba(X)[:, 1]

# Umbral operativo inicial. Puede calibrarse posteriormente según el objetivo institucional.
UMBRAL_RIESGO = 0.50

base_perfiles = X.copy()
base_perfiles["probabilidad_desercion"] = proba_full
base_perfiles = base_perfiles[
    base_perfiles["probabilidad_desercion"] >= UMBRAL_RIESGO
].copy()

print(f"Estudiantes para construir perfiles: {len(base_perfiles)} / {len(X)}")


Estudiantes para construir perfiles: 1405 / 4424


# 5. Construcción de perfiles

El clustering se aplica **solo a estudiantes identificados en riesgo**.

Se conservan las 12 variables utilizadas en el notebook original. A diferencia de la V1, el `scaler_cluster` se guardará junto al KMeans porque es indispensable para asignar correctamente estudiantes nuevos a un cluster.


In [7]:
COLS_CLUSTER = [
    "Age at enrollment",
    "Scholarship holder",
    "Debtor",
    "Tuition fees up to date",
    "Curricular units 1st sem (approved)",
    "Curricular units 1st sem (grade)",
    "Curricular units 2nd sem (approved)",
    "Curricular units 2nd sem (grade)",
    "Displaced",
    "Unemployment rate",
    "Mother's qualification",
    "Father's qualification"
]

scaler_cluster = StandardScaler()
X_cluster_scaled = scaler_cluster.fit_transform(base_perfiles[COLS_CLUSTER])

# Comprobación de k. Se conserva k=4 para mantener la lógica del proyecto,
# pero mostramos silhouette para dejar documentada la decisión.
silhouettes = {}
for k in range(2, 8):
    km_tmp = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    labels_tmp = km_tmp.fit_predict(X_cluster_scaled)
    silhouettes[k] = silhouette_score(X_cluster_scaled, labels_tmp)

print("Silhouette por k:", {k: round(v, 4) for k, v in silhouettes.items()})

K_OPTIMO = 4
kmeans_perfiles = KMeans(
    n_clusters=K_OPTIMO,
    random_state=RANDOM_STATE,
    n_init=20
)
base_perfiles["cluster"] = kmeans_perfiles.fit_predict(X_cluster_scaled)
base_perfiles["cluster"].value_counts().sort_index()


Silhouette por k: {2: np.float64(0.2101), 3: np.float64(0.2194), 4: np.float64(0.145), 5: np.float64(0.1574), 6: np.float64(0.1627), 7: np.float64(0.1674)}


,count
cluster,
0,314
1,387
2,363
3,341


# 6. Analizar los clusters antes de ponerles nombre

**Importante:** KMeans no sabe qué significa un perfil. Los nombres deben salir de las características observadas en cada cluster, no de su número.

Esta tabla compara cada cluster con el promedio de todos los estudiantes en riesgo. Revisar antes de cambiar `CONFIG_PERFILES`.


In [8]:
perfil_clusters = base_perfiles.groupby("cluster")[COLS_CLUSTER].mean().round(2)
promedio_riesgo = base_perfiles[COLS_CLUSTER].mean().round(2)

display(perfil_clusters)
print("\nPromedio de estudiantes en riesgo:")
display(promedio_riesgo.to_frame("promedio"))


,Age at enrollment,Scholarship holder,Debtor,Tuition fees up to date,Curricular units 1st sem (approved),Curricular units 1st sem (grade),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Displaced,Unemployment rate,Mother's qualification,Father's qualification
cluster,,,,,,,,,,,,
0,22.09,0.10,0.26,0.72,4.61,12.29,4.02,12.13,0.61,12.01,6.88,10.90
1,30.82,0.05,0.22,0.60,0.44,2.36,0.06,0.48,0.33,11.33,32.77,33.45
2,22.50,0.09,0.18,0.71,0.75,3.30,0.04,0.33,0.54,12.03,8.27,11.58
3,28.30,0.12,0.24,0.66,4.53,12.04,3.74,11.84,0.40,11.01,34.39,33.00



Promedio de estudiantes en riesgo:


,promedio
Age at enrollment,26.11
Scholarship holder,0.09
Debtor,0.23
Tuition fees up to date,0.67
Curricular units 1st sem (approved),2.45
Curricular units 1st sem (grade),7.17
Curricular units 2nd sem (approved),1.83
Curricular units 2nd sem (grade),5.80
Displaced,0.46
Unemployment rate,11.58


## Configuración semántica de perfiles

Los textos siguientes son **etiquetas configurables**, no conclusiones automáticas de KMeans.

Después de ejecutar la celda anterior, revisa las medias de los cuatro clusters y ajusta estos nombres, factores e intervenciones para que describan lo que realmente muestran tus resultados.

Mientras no se valide esa interpretación, la V2 utiliza nombres neutrales `Perfil 1`, `Perfil 2`, etc., evitando afirmar causas que el clustering por sí solo no demuestra.


In [15]:
CONFIG_PERFILES = {
    0: {
        "perfil": "Rendimiento favorable - estudiante joven",
        "factores_perfil":
            "Rendimiento académico relativamente favorable y menor edad de ingreso.",
        "intervencion_sugerida":
            "Revisar factores no académicos asociados al riesgo y realizar seguimiento preventivo."
    },

    1: {
        "perfil": "Baja progresión académica - estudiante adulto",
        "factores_perfil":
            "Muy baja aprobación y rendimiento académico, junto con una mayor edad de ingreso.",
        "intervencion_sugerida":
            "Priorizar acompañamiento académico y evaluar barreras que puedan estar dificultando la continuidad de estudios."
    },

    2: {
        "perfil": "Baja progresión académica - estudiante joven",
        "factores_perfil":
            "Muy baja aprobación y rendimiento académico durante los primeros semestres.",
        "intervencion_sugerida":
            "Priorizar apoyo académico temprano, tutorías y seguimiento de la progresión curricular."
    },

    3: {
        "perfil": "Rendimiento favorable - estudiante adulto",
        "factores_perfil":
            "Rendimiento académico relativamente favorable y mayor edad de ingreso.",
        "intervencion_sugerida":
            "Revisar factores no académicos asociados al riesgo y mantener seguimiento preventivo."
    }
}


# 7. Diccionarios de entrada de la página

La página recibe las 36 columnas de la plantilla en español y con categorías legibles. Esta capa las transforma a **los códigos UCI originales con los que fue entrenado el modelo**.

Esto es deliberadamente explícito: si aparece una categoría desconocida, el procesamiento falla con un mensaje claro en vez de inventar un código.


In [16]:
COLUMNAS_PAGINA_A_UCI = {
    "Estado civil":"Marital Status",
    "Modo de postulación":"Application mode",
    "Orden de postulación":"Application order",
    "Carrera":"Course",
    "Jornada de asistencia":"Daytime/evening attendance",
    "Formación previa":"Previous qualification",
    "Promedio de formación previa":"Previous qualification (grade)",
    "Nacionalidad":"Nacionality",
    "Nivel educativo de la madre":"Mother's qualification",
    "Nivel educativo del padre":"Father's qualification",
    "Ocupación de la madre":"Mother's occupation",
    "Ocupación del padre":"Father's occupation",
    "Puntaje de admisión":"Admission grade",
    "Desplazado/a":"Displaced",
    "Necesidades educativas especiales":"Educational special needs",
    "Deudor/a":"Debtor",
    "Aranceles al día":"Tuition fees up to date",
    "Género":"Gender",
    "Beneficiario/a de beca":"Scholarship holder",
    "Edad al momento de la matrícula":"Age at enrollment",
    "Estudiante internacional":"International",
    "Asignaturas del 1.er semestre (convalidadas)":"Curricular units 1st sem (credited)",
    "Asignaturas del 1.er semestre (inscritas)":"Curricular units 1st sem (enrolled)",
    "Asignaturas del 1.er semestre (evaluaciones)":"Curricular units 1st sem (evaluations)",
    "Asignaturas del 1.er semestre (aprobadas)":"Curricular units 1st sem (approved)",
    "Promedio del 1.er semestre":"Curricular units 1st sem (grade)",
    "Asignaturas del 1.er semestre (sin evaluaciones)":"Curricular units 1st sem (without evaluations)",
    "Asignaturas del 2.º semestre (convalidadas)":"Curricular units 2nd sem (credited)",
    "Asignaturas del 2.º semestre (inscritas)":"Curricular units 2nd sem (enrolled)",
    "Asignaturas del 2.º semestre (evaluaciones)":"Curricular units 2nd sem (evaluations)",
    "Asignaturas del 2.º semestre (aprobadas)":"Curricular units 2nd sem (approved)",
    "Promedio del 2.º semestre":"Curricular units 2nd sem (grade)",
    "Asignaturas del 2.º semestre (sin evaluaciones)":"Curricular units 2nd sem (without evaluations)",
    "Tasa de desempleo":"Unemployment rate",
    "Tasa de inflación":"Inflation rate",
    "PIB":"GDP"
}

MAPAS_TEXTO_A_CODIGO = {
    "Estado civil": {
        "Soltero/a":1, "Casado/a":2, "Viudo/a":3, "Divorciado/a":4,
        "Unión de hecho":5, "Separado/a legalmente":6
    },
    "Modo de postulación": {
        "1.ª fase - contingente general":1, "Ordenanza N.º 612/93":2,
        "1.ª fase - contingente especial (Azores)":5, "Titular de otro curso superior":7,
        "Ordenanza N.º 854-B/99":10, "Estudiante internacional (licenciatura)":15,
        "1.ª fase - contingente especial (Madeira)":16, "2.ª fase - contingente general":17,
        "3.ª fase - contingente general":18, "Ordenanza N.º 533-A/99, b2 (plan diferente)":26,
        "Ordenanza N.º 533-A/99, b3 (otra institución)":27, "Mayor de 23 años":39,
        "Transferencia":42, "Cambio de carrera":43,
        "Titular de diploma de especialización tecnológica":44,
        "Cambio de institución/carrera":51, "Titular de diploma de ciclo corto":53,
        "Cambio de institución/carrera (internacional)":57
    },
    "Carrera": {
        "Tecnologías de Producción de Biocombustibles":33,
        "Diseño de Animación y Multimedia":171, "Servicio Social (vespertino)":8014,
        "Agronomía":9003, "Diseño de Comunicación":9070, "Enfermería Veterinaria":9085,
        "Ingeniería Informática":9119, "Equinocultura":9130, "Gestión":9147,
        "Servicio Social":9238, "Turismo":9254, "Enfermería":9500,
        "Higiene Oral":9556, "Gestión de Publicidad y Marketing":9670,
        "Periodismo y Comunicación":9773, "Educación Básica":9853,
        "Gestión (vespertino)":9991
    },
    "Jornada de asistencia":{"Vespertina":0, "Diurna":1},
    "Género":{"Femenino":0, "Masculino":1},
}

SI_NO = {"No":0, "Sí":1}
for c in [
    "Desplazado/a", "Necesidades educativas especiales", "Deudor/a",
    "Aranceles al día", "Beneficiario/a de beca", "Estudiante internacional"
]:
    MAPAS_TEXTO_A_CODIGO[c] = SI_NO

# Estas categorías se obtienen automáticamente del dataset UCI para evitar
# mantener manualmente decenas de códigos de educación/ocupación.
# Para producción independiente de UCI, el artefacto guardará los mapas construidos
# a partir de las categorías que use la plantilla.


### Mapas extensos de educación, nacionalidad y ocupación

Para que el notebook sea autocontenido respecto del CSV de prueba creado para Trayectoria, se agregan los mismos textos/códigos usados por la plantilla.


In [17]:
MAPAS_TEXTO_A_CODIGO["Formación previa"] = {
"Educación secundaria":1,"Educación superior - licenciatura/bachelor":2,"Educación superior - grado":3,
"Educación superior - máster":4,"Educación superior - doctorado":5,"Frecuencia de educación superior":6,
"12.º año no completado":9,"11.º año no completado":10,"Otro - 11.º año":12,"10.º año":14,
"10.º año no completado":15,"Educación básica 3.er ciclo (9.º/10.º/11.º) o equivalente":19,
"Educación básica 2.º ciclo (6.º/7.º/8.º) o equivalente":38,"Curso de especialización tecnológica":39,
"Educación superior - grado (1.er ciclo)":40,"Curso técnico superior profesional":42,
"Educación superior - máster (2.º ciclo)":43}

MAPAS_TEXTO_A_CODIGO["Nacionalidad"] = {
"Portuguesa":1,"Alemana":2,"Española":6,"Italiana":11,"Neerlandesa":13,"Inglesa":14,"Lituana":17,
"Angoleña":21,"Caboverdiana":22,"Guineana":24,"Mozambiqueña":25,"Santomense":26,"Turca":32,
"Brasileña":41,"Rumana":62,"Moldava":100,"Mexicana":101,"Ucraniana":103,"Rusa":105,
"Cubana":108,"Colombiana":109}

QUAL = {
"Educación secundaria - 12.º año o equivalente":1,"Educación superior - licenciatura/bachelor":2,
"Educación superior - grado":3,"Educación superior - máster":4,"Educación superior - doctorado":5,
"Frecuencia de educación superior":6,"12.º año no completado":9,"11.º año no completado":10,
"7.º año (antiguo)":11,"Otro - 11.º año":12,"2.º año complementario de secundaria":13,"10.º año":14,
"Curso general de comercio":18,"Educación básica 3.er ciclo (9.º/10.º/11.º) o equivalente":19,
"Curso complementario de secundaria":20,"Curso técnico-profesional":22,
"Curso complementario de secundaria no concluido":25,"7.º año de escolaridad":26,
"2.º ciclo del curso general de secundaria":27,"9.º año no completado":29,"8.º año de escolaridad":30,
"Curso general de Administración y Comercio":31,"Contabilidad y Administración suplementaria":33,
"Desconocido":34,"No sabe leer ni escribir":35,"Sabe leer sin haber cursado 4.º año":36,
"Educación básica 1.er ciclo (4.º/5.º) o equivalente":37,
"Educación básica 2.º ciclo (6.º/7.º/8.º) o equivalente":38,"Curso de especialización tecnológica":39,
"Educación superior - grado (1.er ciclo)":40,"Curso de estudios superiores especializados":41,
"Curso técnico superior profesional":42,"Educación superior - máster (2.º ciclo)":43,
"Educación superior - doctorado (3.er ciclo)":44}
MAPAS_TEXTO_A_CODIGO["Nivel educativo de la madre"] = QUAL
MAPAS_TEXTO_A_CODIGO["Nivel educativo del padre"] = QUAL

OCC = {
"Estudiante":0,"Representantes/dirigentes y directivos":1,
"Especialistas en actividades intelectuales y científicas":2,
"Técnicos y profesionales de nivel intermedio":3,"Personal administrativo":4,
"Trabajadores de servicios personales, seguridad y vendedores":5,
"Agricultores y trabajadores cualificados agropecuarios, pesca y forestal":6,
"Trabajadores cualificados de industria, construcción y artesanía":7,
"Operadores de instalaciones/máquinas y montaje":8,"Trabajadores no cualificados":9,
"Profesiones de las Fuerzas Armadas":10,"Otra situación":90,"Sin especificar":99,
"Oficiales de las Fuerzas Armadas":101,"Sargentos de las Fuerzas Armadas":102,
"Otro personal de las Fuerzas Armadas":103,"Directores de servicios administrativos y comerciales":112,
"Directores de hotelería, restauración, comercio y otros servicios":114,
"Especialistas en ciencias físicas, matemáticas, ingeniería y afines":121,
"Profesionales de la salud":122,"Docentes":123,
"Especialistas en finanzas, contabilidad, organización administrativa y relaciones públicas/comerciales":124,
"Especialistas TIC":125,"Técnicos intermedios de ciencia e ingeniería":131,
"Técnicos y profesionales intermedios de salud":132,
"Técnicos intermedios de servicios jurídicos, sociales, deportivos y culturales":134,
"Técnicos de tecnologías de información y comunicación":135,
"Oficinistas, secretarios y operadores de procesamiento de datos":141,
"Operadores de datos, contabilidad, estadística, finanzas y registros":143,
"Otro personal de apoyo administrativo":144,"Trabajadores de servicios personales":151,"Vendedores":152,
"Trabajadores de cuidados personales y afines":153,"Personal de protección y seguridad":154,
"Agricultores y productores cualificados orientados al mercado":161,
"Agricultores, ganaderos, pescadores, cazadores y recolectores de subsistencia":163,
"Trabajadores cualificados de construcción, excepto electricistas":171,
"Trabajadores cualificados de metalurgia y afines":172,
"Trabajadores cualificados de impresión, precisión, joyería y artesanía":173,
"Trabajadores cualificados de electricidad y electrónica":174,
"Trabajadores de procesamiento de alimentos, madera, confección y otras industrias/artesanías":175,
"Operadores de instalaciones y máquinas fijas":181,"Trabajadores de montaje":182,
"Conductores de vehículos y operadores de equipos móviles":183,"Trabajadores de limpieza":191,
"Trabajadores no cualificados agropecuarios, pesca y forestal":192,
"Trabajadores no cualificados de industria extractiva, construcción, manufactura y transporte":193,
"Ayudantes de preparación de comidas":194,
"Vendedores ambulantes y prestadores de servicios callejeros":195}
MAPAS_TEXTO_A_CODIGO["Ocupación de la madre"] = OCC
MAPAS_TEXTO_A_CODIGO["Ocupación del padre"] = OCC


# 8. Funciones de integración

Aqui se encuentran los datos para coneccion con pagina web

### Función principal
```python
resultado = procesar_csv("archivo_subido.csv")
```

Devuelve un `DataFrame` con todas las columnas originales y agrega:

- `probabilidad_desercion`: número entre 0 y 1.
- `porcentaje_riesgo`: porcentaje de 0 a 100.
- `nivel_riesgo`: Bajo / Medio / Alto.
- `en_riesgo`: Sí / No según el umbral operativo.
- `cluster`: cluster KMeans solo si está en riesgo.
- `perfil`: nombre configurable del perfil.
- `factores_perfil`.
- `intervencion_sugerida`.

**Nota:** Bajo/Medio/Alto es una presentación operativa de la probabilidad, no una nueva predicción del modelo.


In [23]:
def convertir_entrada_a_uci(df_entrada, artefacto):
    """
    Convierte el DataFrame recibido desde la página al formato
    numérico exacto utilizado para entrenar el modelo UCI.

    La página realiza la validación principal.
    Aquí solo se realizan comprobaciones de seguridad antes de predecir.
    """

    df = df_entrada.copy()

    columnas_esperadas = list(
        artefacto["columnas_pagina_a_uci"].keys()
    )

    # 1. Comprobar que existan todas las columnas necesarias
    faltantes = [
        col for col in columnas_esperadas
        if col not in df.columns
    ]

    if faltantes:
        raise ValueError(
            f"Faltan columnas obligatorias: {faltantes}"
        )

    # Trabajamos solamente con las columnas necesarias
    # y en el orden esperado.
    df = df[columnas_esperadas].copy()

    # 2. Comprobar valores faltantes
    if df.isna().any().any():

        errores = {}

        for columna in df.columns:
            filas = df.index[df[columna].isna()].tolist()

            if filas:
                errores[columna] = filas[:10]

        raise ValueError(
            "El dataset contiene valores faltantes. "
            f"Columnas y filas afectadas: {errores}"
        )

    # 3. Convertir categorías de texto a códigos UCI
    for columna, mapa in artefacto["mapas_texto_a_codigo"].items():

        valores_validos = set(mapa.keys())

        valores_recibidos = set(
            df[columna].dropna().astype(str)
        )

        desconocidos = sorted(
            valores_recibidos - valores_validos
        )

        if desconocidos:
            raise ValueError(
                f"Valores no reconocidos en '{columna}': "
                f"{desconocidos[:10]}"
            )

        df[columna] = df[columna].map(mapa)

    # 4. Cambiar nombres españoles por nombres originales UCI
    df = df.rename(
        columns=artefacto["columnas_pagina_a_uci"]
    )

    # 5. Comprobar que todas las variables sean numéricas
    for columna in df.columns:

        try:
            df[columna] = pd.to_numeric(
                df[columna],
                errors="raise"
            )

        except Exception:
            raise ValueError(
                f"La columna '{columna}' contiene "
                "valores que no pueden convertirse a número."
            )

    # 6. Orden exacto utilizado durante entrenamiento
    df = df[artefacto["features_modelo"]]

    # Comprobación final
    if df.isna().any().any():
        raise ValueError(
            "Se generaron valores nulos durante "
            "la transformación de los datos."
        )

    return df


def nivel_riesgo(probabilidad):
    # Rangos operativos para visualización.
    # El umbral real de clasificación sigue siendo UMBRAL_RIESGO.
    if probabilidad < 0.30:
        return "Bajo"
    elif probabilidad < 0.50:
        return "Medio"
    return "Alto"


def enriquecer_dataframe(df_entrada, artefacto):
    """
    Predice riesgo y asigna perfil a los alumnos en riesgo.
    Conserva las columnas originales de entrada.
    """
    X_nuevo = convertir_entrada_a_uci(df_entrada, artefacto)

    modelo = artefacto["modelo_riesgo"]
    probas = modelo.predict_proba(X_nuevo)[:, 1]

    salida = df_entrada.copy()
    salida["probabilidad_desercion"] = probas.round(6)
    salida["porcentaje_riesgo"] = (probas * 100).round(2)
    salida["nivel_riesgo"] = [nivel_riesgo(p) for p in probas]
    salida["en_riesgo"] = np.where(
        probas >= artefacto["umbral_riesgo"], "Sí", "No"
    )

    salida["cluster"] = pd.Series(pd.NA, index=salida.index, dtype="Int64")
    salida["perfil"] = "Sin perfil de riesgo"
    salida["factores_perfil"] = ""
    salida["intervencion_sugerida"] = ""

    mascara = probas >= artefacto["umbral_riesgo"]

    if mascara.any():
        Xc = X_nuevo.loc[mascara, artefacto["cols_cluster"]]
        Xc_scaled = artefacto["scaler_cluster"].transform(Xc)
        clusters = artefacto["kmeans_perfiles"].predict(Xc_scaled)

        idxs = salida.index[mascara]
        salida.loc[idxs, "cluster"] = clusters

        for idx, cluster in zip(idxs, clusters):
            config = artefacto["config_perfiles"][int(cluster)]
            salida.at[idx, "perfil"] = config["perfil"]
            salida.at[idx, "factores_perfil"] = config["factores_perfil"]
            salida.at[idx, "intervencion_sugerida"] = config["intervencion_sugerida"]

    return salida


def procesar_dataframe(df_entrada, ruta_artefacto="trayectoria_pipeline_v2.pkl"):
    """
    Función principal para integración con el backend.

    Recibe:
        DataFrame validado por la página.

    Retorna:
        DataFrame original enriquecido con
        riesgo de deserción y perfil.
    """

    artefacto = joblib.load(ruta_artefacto)

    resultado = enriquecer_dataframe(
        df_entrada,
        artefacto
    )

    return resultado


def procesar_csv(
    ruta_csv,
    ruta_artefacto="trayectoria_pipeline_v2.pkl",
    ruta_salida=None
    ):
    """
    Procesa directamente un archivo CSV.
    """

    entrada = pd.read_csv(
        ruta_csv,
        sep=None,
        engine="python"
    )

    resultado = procesar_dataframe(
        entrada,
        ruta_artefacto
    )

    if ruta_salida is not None:

        resultado.to_csv(
            ruta_salida,
            index=False,
            encoding="utf-8-sig"
        )

    return resultado


# 9. Guardar UN artefacto de producción

En vez de entregar varios `.pkl` separados, se guarda un único archivo con:

- Random Forest.
- KMeans.
- scaler del clustering.
- orden de las 36 variables.
- diccionarios de conversión.
- umbral de riesgo.
- configuración de perfiles.

Así se reduce el riesgo de que la página cargue versiones incompatibles entre sí.


In [24]:
artefacto_produccion = {
    "version": "2.0",
    "modelo_riesgo": modelo_riesgo,
    "kmeans_perfiles": kmeans_perfiles,
    "scaler_cluster": scaler_cluster,
    "features_modelo": FEATURES_MODELO,
    "cols_cluster": COLS_CLUSTER,
    "umbral_riesgo": UMBRAL_RIESGO,
    "columnas_pagina_a_uci": COLUMNAS_PAGINA_A_UCI,
    "mapas_texto_a_codigo": MAPAS_TEXTO_A_CODIGO,
    "config_perfiles": CONFIG_PERFILES
}

joblib.dump(artefacto_produccion, "trayectoria_pipeline_v2.pkl")
print("Guardado: trayectoria_pipeline_v2.pkl")


Guardado: trayectoria_pipeline_v2.pkl


# 10. Prueba end-to-end con el CSV de la página

Sube/ubica `dataset_portugal_formato_trayectoria.csv` en el entorno y ejecuta esta celda.

Si funciona, demuestra el mismo flujo que utilizará la aplicación: **CSV legible → transformación → modelo → perfiles → CSV enriquecido**.


In [25]:
RUTA_PRUEBA = "dataset_portugal_formato_trayectoria.csv"
RUTA_SALIDA = "dataset_portugal_enriquecido.csv"

if Path(RUTA_PRUEBA).exists():
    resultado = procesar_csv(
        RUTA_PRUEBA,
        ruta_artefacto="trayectoria_pipeline_v2.pkl",
        ruta_salida=RUTA_SALIDA
    )

    print("Resultado:", resultado.shape)
    display(resultado.head())
    print("\nNiveles de riesgo:")
    print(resultado["nivel_riesgo"].value_counts())
    print("\nPerfiles:")
    print(resultado["perfil"].value_counts())
    print(f"\nCSV generado: {RUTA_SALIDA}")
else:
    print(
        f"No se encontró {RUTA_PRUEBA}. "
        "Súbelo a Colab y vuelve a ejecutar esta celda."
    )


No se encontró dataset_portugal_formato_trayectoria.csv. Súbelo a Colab y vuelve a ejecutar esta celda.


# 11. Contrato simple para el equipo web

Una vez generado `trayectoria_pipeline_v2.pkl`, **la página no necesita volver a entrenar el modelo**.

El backend solo necesita:

```python
resultado = procesar_csv(
    ruta_del_archivo_subido,
    ruta_artefacto="trayectoria_pipeline_v2.pkl",
    ruta_salida="resultado.csv"
)
```

Para gráficos pueden utilizar directamente, entre otras:

- `resultado["nivel_riesgo"]`
- `resultado["porcentaje_riesgo"]`
- `resultado["perfil"]`
- `resultado["en_riesgo"]`

También conservan las 36 columnas originales, por lo que pueden cruzar riesgo/perfil con carrera, género, jornada, edad, rendimiento, etc.

## Importante para producción
El `.ipynb` sirve para entrenamiento, documentación y pruebas. En una aplicación desplegada es preferible extraer las funciones de integración a un módulo `.py` y cargar el `.pkl` una sola vez al iniciar el servidor.
